<a href="https://colab.research.google.com/github/DanielHevdeli/hafifot-tiug/blob/main/LLM_as_annotator.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!git clone https://github.com/DanielHevdeli/hafifot-tiug.git

Cloning into 'hafifot-tiug'...
remote: Enumerating objects: 138, done.
remote: Counting objects: 100% (138/138), done.
remote: Compressing objects: 100% (126/126), done.
remote: Total 138 (delta 64), reused 38 (delta 6), pack-reused 0 (from 0)
Receiving objects: 100% (138/138), 7.71 MiB | 7.57 MiB/s, done.
Resolving deltas: 100% (64/64), done.


In [ ]:
pip install dspy

In [2]:
import pandas as pd
import os
from typing import Literal, List
import dspy
import requests
import json

ModuleNotFoundError: No module named 'dspy'

In [54]:
posts_df = pd.read_csv('./hafifot-tiug/data/split_data/present.csv')

In [55]:
posts_df.head(2)

,question_id,length,date,text
0,114678,903,2016-04-10 20:05:00,"שלום , אני מאוד מקווה שתוכלו לעזור לי אני לא י..."
1,116349,888,2016-04-24 19:56:00,היי כולם \nיש לי בעיה הקשורה לתספורת שלי. \nאמ...


In [56]:
posts_df.iloc[3]['text']

'בערך מגיל 17 אני סובלת מחרדה, כאשר חלה החרפה קיצונית בגיל 18 עקב לימודים ומצב חברתי גרוע והשפלות יומיומיות  . בהתחלה לא הבנתי בכלל שמדובר בחרדה כי הסימנים היו פיזיולוגים והחששות תמיד היו. התחלתי "להתעלף"  , הרגשתי שאני לא מצליחה לנשום וגם כשאני כן מצליחה אז האוויר לא נכנס לי לריאות , היו לי סחרחורות שהייתי בטוחה שאני עומדת למות ולהתמוטט. נשלחתי למיון וכאשר לא מצאו לי כלום הניחו שזה חרדה.ככה העברתי את השנה האחרונה שלי בבית ספר, כיתה יב. כלומר, במיטה ובדיכאון ללא חברים ולומדת לבגרויות. \nכמה חודשים אחר כך עליתי על מדים והאמת שמצבי השתפר פלאים. אומנם החרדה לא נעלמה , ותמיד היו ניצוצות של חרדה אבל הצלחתי לעבור אותן. \nכהשתחררתי היה לי מין תהום ענקית , הרגשתי בחופש ורציתי כל היום לישון,מה שבצבא לא התאפשר לי וגם כן לא לפני . יצאתי מדי פעם עם חברות אבל גם זה אחכ הפסיק. אחרי כמה חודשים כאלה התחלתי את הפסיכומטרי ולצערי בגלל תחושות של חרדה והתמוטטות בשיעור דחיתי את המועד. אחרי הדחייה נכנסתי לדכאון וחרדה קשים , לא רציתי לצאת מהמיטה.אם פעם התקפים באו והלכו אז המצב שלי בחודשיים האחרונים היו 24/7. 

In [57]:
posts_df.iloc[2]['text']

'הגעתי לאתר הזה במקרה אחרי שקראתי טיפה מפוסטים של אנונימים פה—אני מקווה שגם אוכל למצוא פה מענה או כיוון לפתרון לבעיה שלי.  \nאני לא יודעת כל כך מאיפה להתחיל או מאיפה התחיל הסיפור אבל אני סובלת בערך כל חיי מחרדה ודיכאון. \nהייתי מודעת לעניין אבל מעולם לא חוויתי מצב קיצוני של חרדה משתקת לחלוטין. היו לי התקפים מזעזעים שבגללם הלכתי למיון.  \nאבל בקיצור, בתקופה האחרונה של כמעט שנה אני חוויתי אגרופוביה רצינית עם התקפי דכאון ומחשבות מזעזעות. ברמה של סיעוד. לא התקלחתי, כל היום הייתי ישנה ומסוגרת מתחת לשמיכה. \nהפסיכאטר נתן לי ציפרלקס .  \nאני מרגישה שיפור עם הדיכאון יחסית, ועם ההתקפי חרדה אבל עד היום האגרובופיה לא נעלמה, קשה לי לתפקד. אני רוצה לצאת החוצה,להכיר אנשים, לעבוד, לקבל רישיון, ללמוד. לא להיות למה שהפכתי. \nמיותר לציין,שניתקתי קשר עם כל סבוביי כך שאין לי חיי חברה.  \nאת כל הכסף שאין לי אני מוציאה על מוניות בדרך לפסיכאטר ובחזרה מימנו כי קשה לי לעלות על אוטובוסים. \nהפסקתי עם השיעורי נהיגה בגלל ההתקפים בנהיגה .  \nבקיצור,אם המצב ימשיך אני לא יודעת איך אוכל לשרוד ככה,גם מבחינה כספית וגם 

Let's try to classify each post to either **suicidal-risk** or **non-suicidal-risk**. It may help the publishers of the website to offer first-help to writers of post categorized as suicidal even before other people answer them.

# Annotate Posts

# wait

In [109]:
MODEL_NAME = 'please_how_to_configure_slm'

In [110]:
class SRClassification(dspy.Signature):
    text: str = dspy.InputField(desc="Hebrew post to classify.")
    label: Literal["suicidal-risk", "non-suicidal-risk"] = dspy.OutputField(
        desc="Classification result."
    )

In [111]:
class SRClassifier(dspy.Module):
    def __init__(self):
        super().__init__()
        self.predict = dspy.Predict(SRClassification)

    def forward(self, text: str) -> str:
        result = self.predict(text=text)
        return result.label

In [112]:
sr_classifier = SRClassifier()
posts_labels = []
i = 0
for index, row in posts_df.iterrows():
    if i > 0: break
    question_id = row['question_id']
    text = row['text']

    label = sr_classifier(text)
    posts_labels.append({'question_id': question_id, 'label': label})
    i += 1

print(f"Classification of {len(posts_labels)} posts completed.")

RateLimitError: litellm.RateLimitError: RateLimitError: OpenAIException - You exceeded your current quota, please check your plan and billing details. For more information on this error, read the docs: https://platform.openai.com/docs/guides/error-codes/api-errors.

In [ ]:
post_labels_df = pd.DataFrame(posts_labels)
post_labels_df.head()
save_path = f'./hafifot-tiug/data/labels/present/{MODEL_SHORT_NAME}'
pd.to_csv(f'{save_path}.csv', index=False)
print(f"{MODEL_SHORT_NAME} labels saved to {save_path} successfully.")